In [3]:
import logging
from datetime import date

logging.basicConfig(
    filename="psur_log.txt",
    level=logging.DEBUG,
    format="%(asctime)s — %(levelname)s — %(message)s"
)


class InsufficientReportingError(Exception):
    pass


class PSUR_generator():
    def __init__(self,drug_name, marketing_authorization_date, reporting_period , adverse_events , total_patients_exposed):
        self.drug_name = drug_name
        self.marketing_authorization_date = marketing_authorization_date
        self.reporting_period = reporting_period
        self.adverse_events = adverse_events
        self.total_patients_exposed = total_patients_exposed

    def calculate_exposure_rate(self):
        return self.adverse_events / self.total_patients_exposed * 100
    
    def severity_breakdown(self):
        breakdown = {"mild" : 0 , "moderate" : 0 , "severe" : 0}
        for adr in self.adverse_events:
            sv = adr["severity"]
            if sv in breakdown:
                breakdown[sv] += adr["count"]
        return breakdown
    
    def benefit_risk_assessment(self):
        rate = self.calculate_exposure_rate()
        if rate > 5.0:
            return "⚠ Unfavorable — Detailed Review Required"
        return "Favorable — Benefit outweighs Risk"

    
    def top_events(self, n = 3):
        sorted_events = sorted(self.adverse_events, key=lambda x : x["count"], reverse=True)
        return sorted_events[:n]
    
    def calculate_exposure_rate(self):
        total_ae = sum(adr["count"]for adr in self.adverse_events)
        rate = total_ae / self.total_patients_exposed * 100
        return round(rate, 2)
    
    def generate_psur(self):
        total_ae = sum(adr["count"]for adr in self.adverse_events)

        if total_ae < 10:
            raise InsufficientReportingError(f"Insufficient reporting for PSUR generation. | minimum 10 required")
        
        rate     = self.calculate_exposure_rate()
        breakdown = self.severity_breakdown()
        top3     = self.top_events(3)
        br       = self.benefit_risk_assessment()

        print(f"\n{'='*60}")
        print(f"PERIODIC SAETY UPDATE REPORT")
        print(f"ICH FORMATED REPORT")
        print(f"\n{'='*60}")
        print(f"  Drug           : {self.drug_name}")
        print(f"  Auth Date      : {self.marketing_authorization_date}")
        print(f"  Period         : {self.reporting_period[0]}"
              f" to {self.reporting_period[1]}")
        print(f"  Patients       : {self.total_patients_exposed:,}")
        print(f"  Exposure Rate  : {rate}%")
        print(f"\n  Severity Breakdown:")
        for sev, count in breakdown.items():
            bar = "█" * (count // 5)
            print(f"    {sev:<10}: {count:>4}  {bar}")
        print(f"\n  Top {len(top3)} Reported Events:")
        for i, ae in enumerate(top3, 1):
            print(f"    {i}. {ae['event']:<25} — "
                  f"{ae['count']} cases ({ae['severity']})")
        print(f"\n  Benefit-Risk   : {br}")
        print(f"{'='*60}")

        with open("psur_report.txt", "a") as f:
            f.write("=" * 60 + "\n")
            f.write("PERIODIC SAFETY UPDATE REPORT\n")
            f.write(f"Drug     : {self.drug_name}\n")
            f.write(f"Period   : {self.reporting_period[0]}"
                    f" to {self.reporting_period[1]}\n")
            f.write(f"Patients : {self.total_patients_exposed:,}\n")
            f.write(f"AE Rate  : {rate}%\n")
            f.write(f"Severity : {breakdown}\n")
            f.write("Top Events:\n")
            for ae in top3:
                f.write(f"  - {ae['event']}: {ae['count']}\n")
            f.write(f"B/R      : {br}\n")
            f.write("=" * 60 + "\n\n")

        print(f"  📄 Saved to psur_report.txt")
        logging.info(
            f"PSUR GENERATED | {self.drug_name} | "
            f"Rate={rate}% | {br}")

    def __str__(self):
        return (f"{self.drug_name} | "
                f"{self.reporting_period[0]} — "
                f"{self.reporting_period[1]}")

    
    

#lets try with examples:
drugs = [
    PSUR_generator(
        drug_name            = "Metformin HCl 500mg",
        marketing_authorization_date  = "2020-03-15",
        reporting_period     = ("2025-01-01", "2025-12-31"),
        adverse_events = [
            {"event": "Nausea",          "count": 45, "severity": "Mild"},
            {"event": "Diarrhea",        "count": 38, "severity": "Mild"},
            {"event": "Lactic Acidosis", "count": 8,  "severity": "Severe"},
            {"event": "Headache",        "count": 22, "severity": "Mild"},
            {"event": "Hypoglycemia",    "count": 15, "severity": "Moderate"},
            {"event": "Vomiting",        "count": 12, "severity": "Moderate"},
        ],
        total_patients_exposed = 5000
    ),

    PSUR_generator(
        drug_name            = "Warfarin 5mg",
        marketing_authorization_date  = "2018-06-20",
        reporting_period     = ("2025-01-01", "2025-12-31"),
        adverse_events = [
            {"event": "Bleeding",        "count": 95, "severity": "Severe"},
            {"event": "Bruising",        "count": 78, "severity": "Moderate"},
            {"event": "Hair Loss",       "count": 42, "severity": "Mild"},
            {"event": "GI Bleeding",     "count": 55, "severity": "Severe"},
            {"event": "Fatigue",         "count": 30, "severity": "Mild"},
        ],
        total_patients_exposed = 3000
    ),
]

for drug in drugs:
    try:
        drug.generate_psur()
    except InsufficientReportingError as e:
        print(e)


        
    


    


PERIODIC SAETY UPDATE REPORT
ICH FORMATED REPORT

  Drug           : Metformin HCl 500mg
  Auth Date      : 2020-03-15
  Period         : 2025-01-01 to 2025-12-31
  Patients       : 5,000
  Exposure Rate  : 2.8%

  Severity Breakdown:
    mild      :    0  
    moderate  :    0  
    severe    :    0  

  Top 3 Reported Events:
    1. Nausea                    — 45 cases (Mild)
    2. Diarrhea                  — 38 cases (Mild)
    3. Headache                  — 22 cases (Mild)

  Benefit-Risk   : Favorable — Benefit outweighs Risk
  📄 Saved to psur_report.txt

PERIODIC SAETY UPDATE REPORT
ICH FORMATED REPORT

  Drug           : Warfarin 5mg
  Auth Date      : 2018-06-20
  Period         : 2025-01-01 to 2025-12-31
  Patients       : 3,000
  Exposure Rate  : 10.0%

  Severity Breakdown:
    mild      :    0  
    moderate  :    0  
    severe    :    0  

  Top 3 Reported Events:
    1. Bleeding                  — 95 cases (Severe)
    2. Bruising                  — 78 cases (Moderate)

PSURGenerator(
        drug_name            = "Metformin HCl 500mg",
        marketing_auth_date  = "2020-03-15",
        reporting_period     = ("2025-01-01", "2025-12-31"),
        adverse_events = [
            {"event": "Nausea",          "count": 45, "severity": "Mild"},
            {"event": "Diarrhea",        "count": 38, "severity": "Mild"},
            {"event": "Lactic Acidosis", "count": 8,  "severity": "Severe"},
            {"event": "Headache",        "count": 22, "severity": "Mild"},
            {"event": "Hypoglycemia",    "count": 15, "severity": "Moderate"},
            {"event": "Vomiting",        "count": 12, "severity": "Moderate"},
        ],
        total_patients_exposed = 5000
    ),